#### Task 3: Average order value per InvoiceNo and invoices with order value above 10,000.

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 
 
def load_data():
    """Load the real OnlineRetail.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("OnlineRetail.csv", "OnlineRetail_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding="ISO-8859-1", dtype={"InvoiceNo": str, "StockCode": str})
            print(f"Loaded {name}: {len(df):,} rows" + ("" if name == "OnlineRetail.csv" else "  (SIMULATED STAND-IN DATA, not the real file)"))
            break
    else:
        raise FileNotFoundError("Put OnlineRetail.csv (or run generate_sample_data.py) in the same folder.")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
    return df
 
 
def valid_rows(df):
    """Drop cancellations (InvoiceNo starts with C) and rows with non-positive quantity or price."""
    return df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
 
 
df = valid_rows(load_data())
 
# Order value = total sales per InvoiceNo (Excel: SUMIFS on the LineTotal column)
orders = df.groupby("InvoiceNo").agg(order_value=("LineTotal", "sum"), country=("Country", "first"))
print(f"\nNumber of invoices = {len(orders):,}")
print(f"Average order value = {orders['order_value'].mean():,.2f}")
print(f"Median order value  = {orders['order_value'].median():,.2f}   (lower than the mean: a few huge orders pull the mean up)")
print(f"Largest order       = {orders['order_value'].max():,.2f}")
 
# Highlight invoices above 10,000 (Excel: Conditional Formatting > Highlight Cells Rules > Greater Than)
THRESHOLD = 10_000
high = orders[orders["order_value"] > THRESHOLD].sort_values("order_value", ascending=False)
print(f"\nInvoices with order value above {THRESHOLD:,}: {len(high)} ({len(high) / len(orders):.2%} of invoices)")
print(high.round(2).head(15).to_string())
print("\nNote: the dataset is in pounds (GBP); the 10,000 threshold from the task is applied to the numbers as given.")

Loaded OnlineRetail_sample.csv: 3,989 rows  (SIMULATED STAND-IN DATA, not the real file)

Number of invoices = 720
Average order value = 435.97
Median order value  = 79.81   (lower than the mean: a few huge orders pull the mean up)
Largest order       = 13,671.14

Invoices with order value above 10,000: 3 (0.42% of invoices)
           order_value         country
InvoiceNo                             
536405        13671.14  United Kingdom
536815        11505.69  United Kingdom
536965        10707.12  United Kingdom

Note: the dataset is in pounds (GBP); the 10,000 threshold from the task is applied to the numbers as given.
